In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Imports
import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[2]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

/opt/anaconda3/envs/reu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
# Inputs
checkpoint_file="../../../saved_models/RandomPos-cenFL.ckpt"
data_file = "../../../data/RandomPos_0709.csv"
train_perc = 80
norm_trained = False


In [22]:
# Load models, define wrappers
model = load_model_checkpoint(checkpoint_file, gpu=False)

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        # ART passes one-hot labels; CrossEntropyLoss needs class indices
        if b.dim() == 3:
            b = b.argmax(dim=-1)  # (batch, seq_len, num_classes) → (batch, seq_len)
        return self.loss(
            a.permute(0, 2, 1),  # (batch, seq_len, C) → (batch, C, seq_len)
            b.long()
        )

class NormalizedCfCWrapper(nn.Module):
    def __init__(self, modena_model, data_min, data_max):
        super().__init__()
        self.modena_model = modena_model
        self.norm_trained = norm_trained
        if not self.norm_trained:
            # MUST wrap in torch.tensor() explicitly — do not assign raw numpy arrays
            self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
            self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))

    def forward(self, x_normalized):        
        if not self.norm_trained:
            x_raw = x_normalized * (self.data_max - self.data_min) + self.data_min
        else:
            x_raw = x_normalized
        logits, _ = self.modena_model(x_raw)
        return logits

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [ ]:
# Load data for og model (very time consuming part)
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc)



In [27]:
# Set model variables
wrapped_model = NormalizedCfCWrapper(modena_model=model.model,
                                     data_min=x_train.amin(dim=(0,1)),
                                     data_max=x_train.amax(dim=(0,1)))
criterion = SequenceCrossEntropy()
optimizer = optim.Adam(
    wrapped_model.parameters(),
    lr=0.001
)

classifier = PyTorchClassifier(
    model=wrapped_model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(10, 7),
    nb_classes=20,
    clip_values=(0.0, 1.0), # for normalized
    device_type="cpu"
)

print(type(model))
print(type(model.model))
print(type(wrapped_model))

<class 'utils.models.OBU'>
<class 'utils.models.Modena'>
<class '__main__.NormalizedCfCWrapper'>


/var/folders/km/lbmvfj6j6pb808sy4csbkgw40000gn/T/ipykernel_57866/909651698.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
/var/folders/km/lbmvfj6j6pb808sy4csbkgw40000gn/T/ipykernel_57866/909651698.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))


In [28]:
# REGULAR: Test the model
print("TESTING |", f"norm_trained: {norm_trained}", "| NO wrapper")
out = model.test(x_test, y_test, mathy=True)

not_same = 0
for i in range(20):
    print(f"=== {i+1}/20 ===")
    out_test = model.test(x_test, y_test, mathy=True)
    if out != out_test:
        print("NOT SAME!")
        not_same +=1
print("# of times with distinct output:", not_same)
print(f"Out: {out}")

TESTING | norm_trained: False | NO wrapper
torch.Size([124774, 10, 20])
0.9643128342104006
0.9797491232802805
Model got 1226792/1247740 right.
Accuracy: 0.9832112459326462, Precision: 0.9643128342104006, Recall: 0.9797491232802805, F1 Score: 0.9719696949422882
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
=== 1/20 ===
torch.Size([124774, 10, 20])
0.9643128342104006
0.9797491232802805
Model got 1226792/1247740 right.
Accuracy: 0.9832112459326462, Precision: 0.9643128342104006, Recall: 0.9797491232802805, F1 Score: 0.9719696949422882
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
=== 2/20 ===
torch.Size([124774, 10, 20])
0.9643128342104006
0.9797491232802805
Model got 1226792/1247740 right.
Accuracy: 0.9832112459326462, Precision: 0.9643128342104006, Recall: 0.9797491232802805, F1 Score: 0.9719696949422882
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
=== 3/20 ===
torch.Size([124774, 10, 20])
0.9643128342104006
0.9797491232802805
Model got 1226792

In [29]:
# Load data at *normalized* (very time consuming part)
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=train_perc)


In [30]:
# Benign test
benign_predictions = classifier.predict(x_test, batch_size=64)
benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
true_flat = y_test.flatten()
pred_flat = benign_pred_classes.flatten()

accuracy = np.sum(benign_pred_classes == y_test.numpy()) / benign_pred_classes.size
precision =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

print(f"Benign accuracy: {accuracy}")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

not_same = 0
for i in range(20):
    print(f"=== {i+1}/20 ===")
    benign_predictions = classifier.predict(x_test, batch_size=64)
    benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
    true_flat = y_test.flatten()
    pred_flat = benign_pred_classes.flatten()

    accuracy_test = np.sum(benign_pred_classes == y_test.numpy()) / benign_pred_classes.size
    precision_test =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
    recall_test = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
    f1_test = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

    if accuracy == accuracy_test and precision == precision_test \
        and recall == recall_test and f1 == f1_test:
        pass
    else:
        print("NOT SAME! f1:", f1)

print("# of distinct values:", not_same)

Benign accuracy: 0.9832112459326462
Precision: 0.9833399824757991
Recall: 0.9832112459326462
F1: 0.9832494048527688
=== 1/20 ===
=== 2/20 ===
=== 3/20 ===
=== 4/20 ===
=== 5/20 ===
=== 6/20 ===
=== 7/20 ===
=== 8/20 ===
=== 9/20 ===
=== 10/20 ===
=== 11/20 ===
=== 12/20 ===
=== 13/20 ===
=== 14/20 ===
=== 15/20 ===
=== 16/20 ===
=== 17/20 ===
=== 18/20 ===
=== 19/20 ===
=== 20/20 ===
# of distinct values: 0


In [11]:
import os
import json

# Adversarial test function
def adv_test(eps: int, path : str):
    print(f"=== Eps: {eps} ===")
    attack = FastGradientMethod(estimator=classifier, eps=eps)
    
    x_test_adv = attack.generate(x=x_test_norm)#, y=y_test[:fraction])
    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len)
    adversarial_accuracy = np.sum(pred_classes == y_test) / pred_classes.size
    print(f"Adversarial accuracy: {adversarial_accuracy}")

    true_flat = y_test.flatten()
    pred_flat = pred_classes.flatten()
    precision = precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
    recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    # Save metrics to JSON in ./output
    os.makedirs(path, exist_ok=True)
    metrics = {
        "eps": eps,
        "accuracy": float(adversarial_accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    output_path = f"{path}/adv_test_eps_{eps}.json"
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Saved metrics to {output_path}")

    return metrics

In [12]:
# epsilon sweep for PGD
for i in range(1, 31, 1):
    eps = float(i/100)
    adv_test(eps, "./output/norm-fgsm")


=== Eps: 0.01 ===
Adversarial accuracy: 0.9935819962492186
Precision: 0.993600170229911
Recall: 0.9935819962492186
F1: 0.9935681359006094
Saved metrics to ./output/norm-fgsm/adv_test_eps_0.01.json
=== Eps: 0.02 ===
Adversarial accuracy: 0.9920400083350698
Precision: 0.9920342308462028
Recall: 0.9920400083350698
F1: 0.99203165981669
Saved metrics to ./output/norm-fgsm/adv_test_eps_0.02.json
=== Eps: 0.03 ===
Adversarial accuracy: 0.9773799028643788
Precision: 0.9780251118846397
Recall: 0.9773799028643788
F1: 0.9775150975427012
Saved metrics to ./output/norm-fgsm/adv_test_eps_0.03.json
=== Eps: 0.04 ===
Adversarial accuracy: 0.9518753907063972
Precision: 0.9566067246271227
Recall: 0.9518753907063972
F1: 0.9526658351218236
Saved metrics to ./output/norm-fgsm/adv_test_eps_0.04.json
=== Eps: 0.05 ===
Adversarial accuracy: 0.9091477391123151
Precision: 0.9272440148546851
Recall: 0.9091477391123151
F1: 0.9118417306404261
Saved metrics to ./output/norm-fgsm/adv_test_eps_0.05.json
=== Eps: 0.06

Normalized:
Model got 1226792/1247740 right.
Accuracy: 0.9832112459326462, Precision: 0.9643128342104006, Recall: 0.9797491232802805, F1 Score: 0.9719696949422882
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Saved backup of main model.
SAVE PATH: out/FL/RandPos-Normalized-False-20-30-30-200/mainModelBackup.ckpt
Elapsed time (ns): 20533253865914